# CSC 4792 Group 36 — Samfya Town Council Web Scraping
**Role:** Person A — Scraper Lead  
**Council:** Samfya Town Council  
**Base URL:** https://www.samfyacouncil.gov.zm

This notebook documents the raw data acquisition stage. Cleaning and standardisation are deliberately left for Person B. All CSV outputs are pipe-separated (`|`), retain source URLs, and record retrieval timestamps.

## 1. Data sources
The scraper targets: CDF community projects, grants and loans, skills/bursaries, publications (IDP, minutes, budgets and financial statements), and council news posts.

Important page URLs are configured in `scripts/samfya_scraper.py`.

## Data Cleaning & Preprocessing (Person B)

Person A produced pipe-delimited raw CSVs in `data/raw/`. This section documents the cleaning pipeline implemented in `scripts/clean_data.py`. The pipeline is idempotent and can be re-run at any time. All rules, assumptions, and limitations are recorded in `CLEANING_NOTES.md` at the repository root.

The cleaning step produces three final datasets in `data/cleaned/`:

| File | Rows | Cols | One row = |
|---|---|---|---|
| `db-unza26-csc4792-samfya_cdf_projects.csv` | 28 | 30 | one reported CDF project or funding activity |
| `db-unza26-csc4792-samfya_council_resources.csv` | 90 | 17 | one council document or resource |
| `db-unza26-csc4792-samfya_news.csv` | 21 | 20 | one council news article |

In [1]:
from pathlib import Path
import pandas as pd

CWD = Path.cwd()
ROOT = CWD if (CWD / "data").exists() else CWD.parent
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "cleaned"

print("Project root :", ROOT.resolve())
print("Raw files    :", len(list(RAW.glob("*.csv"))))
print("Cleaned files:", len(list(CLEAN.glob("*.csv"))))

Project root : C:\Users\Pastor Athanase\group36-data-mining-project
Raw files    : 8
Cleaned files: 3


In [2]:
cdf       = pd.read_csv(CLEAN / "db-unza26-csc4792-samfya_cdf_projects.csv", sep="|")
resources = pd.read_csv(CLEAN / "db-unza26-csc4792-samfya_council_resources.csv", sep="|")
news      = pd.read_csv(CLEAN / "db-unza26-csc4792-samfya_news.csv", sep="|")

print("CDF projects     :", cdf.shape)
print("Council resources:", resources.shape)
print("News             :", news.shape)

CDF projects     : (28, 30)
Council resources: (90, 17)
News             : (21, 20)


In [3]:
cdf[[
    "record_kind", "council_clean", "ward_clean",
    "project_name_clean", "status_clean", "year_clean",
    "amount_zmw_raw_kept", "amount_zmw", "source_url_clean"
]].head(10)

,record_kind,council_clean,ward_clean,project_name_clean,status_clean,year_clean,amount_zmw_raw_kept,amount_zmw,source_url_clean
0,project,Samfya Town Council,Kapilibila,Construction Of 1×3 Classroom Block At Kaishe ...,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
1,project,Samfya Town Council,Lumamya,Construction Of 1×3 Classroom Block At Pwele P...,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
2,project,Samfya Town Council,Musaba,Construction Of 1×3 Classroom Block At Yamba P...,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
3,project,Samfya Town Council,Mano,Construction Of Mano Health Post At Mano,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
4,project,Samfya Town Council,Kapata,Construction Of Kansenga Ablution At Kansenga ...,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
5,project,Samfya Town Council,Kapata,Construction Of Kansenga Health Post,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
6,project,Samfya Town Council,Lupili,"Water, Sanitation And Solar Installations",Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
7,project,Samfya Town Council,Lupili,Veronica Chikonde Rural Health Center,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
8,project,Samfya Town Council,Various Wards,Procurement Of Boats,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...
9,project,Samfya Town Council,All Wards,Procurement Of Bicycles,Under procurement,2025,NaN,NaN,https://www.samfyacouncil.gov.zm/wp-content/up...


In [4]:
cdf["amount_zmw_num"] = pd.to_numeric(cdf["amount_zmw"], errors="coerce")

print("Rows with a numeric amount:", cdf["amount_zmw_num"].notna().sum(), "of", len(cdf))
print("Total disbursed (sum of numeric amounts):",
      f"{cdf['amount_zmw_num'].sum():,.2f} ZMW")
print()
print("Rows with a parsed amount:")
print(cdf[cdf["amount_zmw_num"].notna()][
    ["project_name_clean", "amount_zmw_raw_kept", "amount_zmw_num"]
].to_string(index=False))

Rows with a numeric amount: 7 of 28
Total disbursed (sum of numeric amounts): 58,037,000.00 ZMW

Rows with a parsed amount:
                                                                                                                        project_name_clean amount_zmw_raw_kept  amount_zmw_num
                                                                                                                Mabo Kunda maternity annex             2256000       2256000.0
                                                                                             Chinsanka Rural Health Center maternity annex             1181000       1181000.0
                                  CDF disbursement to 140 community groups; cumulative beneficiary groups reported as over 600 since 2022.             2800000       2800000.0
   Article reports over K3 million annual allocation to CDF roads component; 26 km township roads concluded and 30 km feeder roads opened.            >3000000       3000000.0
 

In [5]:
print("Project/activity status distribution:")
print(cdf["status_clean"].value_counts(dropna=False).to_string())

Project/activity status distribution:
status_clean
Unknown              15
Under procurement    13


In [6]:
cdf_missing_url  = (cdf["source_url_clean"].fillna("") == "").sum()
res_missing_url  = (resources["document_url_clean"].fillna("") == "").sum()
news_missing_url = (news["article_url_clean"].fillna("") == "").sum()

print("CDF rows missing source_url_clean       :", cdf_missing_url, "/", len(cdf))
print("Resource rows missing document_url_clean:", res_missing_url, "/", len(resources))
print("News rows missing article_url_clean     :", news_missing_url, "/", len(news))

CDF rows missing source_url_clean       : 0 / 28
Resource rows missing document_url_clean: 0 / 90
News rows missing article_url_clean     : 0 / 21


In [7]:
news[[
    "record_id", "title_clean", "published_date_clean",
    "mentioned_amount_zmw", "article_url_clean"
]].head(10)

,record_id,title_clean,published_date_clean,mentioned_amount_zmw,article_url_clean
0,NEWS0001,ONE HUNDRED AND FOURTY GROUPS BAGS CDF GRANTS,2026-09-03,2800000.0,https://www.samfyacouncil.gov.zm/?p=2841
1,NEWS0002,A 30 MILLION KWACHA HARBOR COMPLETED AND READY...,2026-09-03,30000000.0,https://www.samfyacouncil.gov.zm/?p=2839
2,NEWS0003,GOVERNMENT BACKS SAMFYA INVESTMENT POTENTIAL W...,2026-09-03,NaN,https://www.samfyacouncil.gov.zm/?p=2837
3,NEWS0004,TOWNSHIP ROAD MAINTAINANCE SOLIDIFIED WITH GOV...,2026-09-03,2300000.0,https://www.samfyacouncil.gov.zm/?p=2834
4,NEWS0005,Traditional leaders in Samfya rally behind upc...,2026-04-12,NaN,https://www.samfyacouncil.gov.zm/?p=2689
5,NEWS0006,SAMFYA YOUTHS APPLAUDS GOVERNMENT FOR PROFFESS...,2026-02-23,NaN,https://www.samfyacouncil.gov.zm/?p=2679
6,NEWS0007,SAMFYA TOWN COUNCIL AND NORTEC FORMS A PACT,2026-02-11,NaN,https://www.samfyacouncil.gov.zm/?p=2669
7,NEWS0008,SAMFYA DISTRICT SEES BOOST IN FEEDER ROADS CON...,2025-12-09,NaN,https://www.samfyacouncil.gov.zm/?p=2429
8,NEWS0009,SAMFYA YOUTHFUL ENTREPRENUERS MANIFEST QUALITY...,2025-12-09,NaN,https://www.samfyacouncil.gov.zm/?p=2416
9,NEWS0010,SAMFYA ENTREPRENUERS IN FINANCIAL HEAVE,2025-12-09,NaN,https://www.samfyacouncil.gov.zm/?p=2364


### Cleaning rules summary

- Whitespace trimmed and collapsed; non-breaking spaces normalised.
- Council name standardised to `Samfya Town Council`.
- Wards mapped via a curated dictionary (`ward_clean`); unmapped wards kept as title-cased free text.
- Project statuses mapped to: `Completed`, `In progress`, `Under procurement`, `Deferred`, `Proposed`, `Unknown`.
- Monetary values parsed to numeric `amount_zmw` (ZMW). Original text preserved in `amount_zmw_raw_kept`. Handles `k`/`m` suffixes and `million`/`billion` words.
- Years extracted via regex `(19|20)\d{2}`. Dates normalised to ISO 8601.
- Exact duplicate rows removed. Likely duplicates across sources retained when they represent different stages, dates, or amounts.
- Provenance columns (`source_url_clean`, `document_url_clean`, `article_url_clean`, `retrieved_at_utc_clean`) preserved on every row.

Full details: see `CLEANING_NOTES.md`.

In [8]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
from samfya_scraper import ScrapeConfig, run


## 2. Run document discovery and news scraping
This performs low-rate requests to public Samfya Town Council pages and writes raw pipe-separated CSV files into `data/raw/`.


In [9]:
resources, news = run(ScrapeConfig(delay_seconds=1.0, max_news_pages=10, download_documents=False, fetch_article_details=True))
display(resources[:5])
display(news[:5])


[resources] community_projects: https://www.samfyacouncil.gov.zm/?page_id=2029
[resources] grants_and_loans: https://www.samfyacouncil.gov.zm/?page_id=2097
[resources] skills_and_bursaries: https://www.samfyacouncil.gov.zm/?page_id=2078
[resources] publications: https://www.samfyacouncil.gov.zm/?page_id=195
[resources] application_forms: https://www.samfyacouncil.gov.zm/?page_id=1224
[news] page 1: https://www.samfyacouncil.gov.zm/?cat=1
[news] page 2: https://www.samfyacouncil.gov.zm/?cat=1&paged=2
[news] page 3: https://www.samfyacouncil.gov.zm/?cat=1&paged=3
[news] page 4: https://www.samfyacouncil.gov.zm/?cat=1&paged=4
Saved 90 resource records and 21 news records.


[{'council': 'Samfya Town Council',
  'category': 'community_projects',
  'year_raw': '',
  'document_title_raw': 'List of Proposed Projects From WDCs By Ward',
  'document_url': 'http://www.samfyacouncil.gov.zm/wp-content/uploads/2025/06/List-of-Proposed-Projects-From-WDCs-By-Ward.pdf',
  'file_type': 'pdf',
  'source_page': 'https://www.samfyacouncil.gov.zm/?page_id=2029',
  'retrieved_at_utc': '2026-09-13T17:05:19+00:00'},
 {'council': 'Samfya Town Council',
  'category': 'community_projects',
  'year_raw': '2025',
  'document_title_raw': 'List Of Approved Community Projects 2025',
  'document_url': 'http://www.samfyacouncil.gov.zm/wp-content/uploads/2025/11/List-Of-Approved-Community-Projects-2025.pdf',
  'file_type': 'pdf',
  'source_page': 'https://www.samfyacouncil.gov.zm/?page_id=2029',
  'retrieved_at_utc': '2026-09-13T17:05:19+00:00'},
 {'council': 'Samfya Town Council',
  'category': 'community_projects',
  'year_raw': '',
  'document_title_raw': 'List of Deffered Projects B

[{'council': 'Samfya Town Council',
  'title_raw': 'ONE HUNDRED AND FOURTY GROUPS BAGS CDF GRANTS',
  'published_date_raw': 'September 3, 2026',
  'excerpt_raw': '2.8-million-kwacha Constituency Development Fund disbursed to one hundred and forty community groups in Samfya district in 2026 bring the accumulative beneficiary groups to over six hundred since 2022. The event officiated by Luapila Province Deputy Permanent Secretary Evans Sikabbuba and witness by traditional leaders in the district, brought the grants limbo to an end as… Continue reading ONE HUNDRED AND FOURTY GROUPS BAGS CDF GRANTS Published September 3, 2026 Categorized as Uncategorized',
  'article_url': 'https://www.samfyacouncil.gov.zm/?p=2841',
  'source_listing_page': 'https://www.samfyacouncil.gov.zm/?cat=1',
  'article_text_raw': '2.8-million-kwacha Constituency Development Fund disbursed to one hundred and forty community groups in Samfya district in 2026 bring the accumulative beneficiary groups to over six hund

## 3. Optional document download
Only enable this after reviewing the resource index. It downloads public council documents at a low request rate.


In [10]:
# Uncomment when required:
# resources, news = run(ScrapeConfig(delay_seconds=1.0, download_documents=True))


## 4. PDF extraction
PDF extraction is optional and has separate packages. First run `pip install -r requirements-pdf.txt`, then run `python scripts/extract_pdf_tables.py`. It writes raw page text and table extracts, a table manifest, and a CDF evidence index. The output can be irregular because council PDFs use different layouts; this is expected at the scraping stage.


## 5. Validate the handoff files
Run the validation before Person B starts cleaning. Empty live outputs are errors; missing evidence URLs are warnings that Person A must resolve or explicitly document.


In [11]:
from validate_raw_data import run as validate_raw_data
validate_raw_data()


Validation report: C:\Users\Pastor Athanase\group36-data-mining-project\data\raw\db-unza26-csc4792-samfya_raw_validation_report.csv
Errors: 0; warnings: 2


0

## 6. Handoff
Person B should clean the raw outputs, standardise fields and remove duplicates while preserving source URLs. See `HANDOFF_TO_PERSON_B.md`.
